# Experiment 1B - Expanded directional model-complexity comparison

Compare the same three synthetic inflammatory truth datasets with four directional model families:

1. **Full NODDI + soma**: neurite + extra-axonal + soma + free water.
2. **Full minus extra-axonal**: neurite + soma + free water.
3. **Reduced anisotropic WM**: one Zeppelin WM compartment + soma + free water.
4. **Reduced isotropic background**: isotropic WM/background + soma + free water.

The primary biological outputs are `r_soma`, `soma_fraction`, and `csf_fraction` (`f_fw`). Every model is fit to the **same noisy directional data** generated from the full NODDI+soma truth model for each scenario and repeat.

The new no-extra family is an explicit intermediate-complexity test: does the extra-axonal compartment actually improve recovery of the soma/free-water biology?

## Experimental design

- Acquisition: `rm_full` initially.
- Scenario 1: `f_soma=0.10`, `r_soma=5`, `f_fw=0.15`, `f_wm=0.75`.
- Scenario 2: `f_soma=0.15`, `r_soma=7`, `f_fw=0.20`, `f_wm=0.65`.
- Scenario 3: `f_soma=0.20`, `r_soma=8`, `f_fw=0.25`, `f_wm=0.55`.
- Increasing soma and free-water fractions are balanced by decreasing WM.
- Default: 50 paired noise repeats, SNR=100, 5 LS starts, max 300 evaluations.
- MCMC is intentionally omitted for the first pass.

In [93]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/.")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from dmri_models import Ball, Component, LeastSquaresFitter, Model, ModelPreset, Parameter, ParameterSpace, Simplex, Sphere, Stick, Zeppelin
from dmri_models.acquisitions import rm_full
from dmri_models.compartments import WatsonFOD, FODConvolved, NODDIExtraNeurite, NODDISomaExtraNeurite, NODDISomaDispersionExtraNeurite


In [94]:
N_REPEATS = 1
BASE_SEED = 1000
N_STARTS = 5
MAX_NFEV = 300
SNR = 100.0
acquisition = rm_full()
acquisition.print_summary()


----------------------------------------------------------------
Acquisition summary
- measurements: 200
- b-value shells (ms/um^2): 7 (tolerance: 0.100)
  - b=0.000: 20 measurements, 0 non-zero directions
  - b=0.500: 30 measurements, 30 non-zero directions
  - b=1.000: 30 measurements, 30 non-zero directions
  - b=2.000: 30 measurements, 30 non-zero directions
  - b=3.000: 30 measurements, 30 non-zero directions
  - b=4.000: 30 measurements, 30 non-zero directions
  - b=5.000: 30 measurements, 30 non-zero directions
- gradient directions: 180 unique non-zero across 180 diffusion-weighted measurements
- b-tensor shape: 1.000
- pulse duration (ms): 5.000
- diffusion time (ms): 12.000
- gradient strength (T/m): 0.000 to 0.520
- echo time (ms): 26.000
- inversion time: n/a
- repetition time (ms): 2500.000
----------------------------------------------------------------


## 1. Model definitions

The new model family is **full minus extra-axonal**: neurite (stick + Watson FOD) + soma + free water. It sits between the full NODDI+soma model and the single-background reduced model.

In [95]:
def full_noddi_soma_model(*, theta=1.0, phi=1.2, vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_perp=False, vary_d_csf=False, fod_quadrature_theta=20, fod_quadrature_phi=40):
    fod = WatsonFOD(kappa="kappa", theta="theta", phi="phi", n_theta=fod_quadrature_theta, n_phi=fod_quadrature_phi)
    model = Model([
        Component("neurite", compartment=FODConvolved(response=Stick(diffusivity="d_in", orientation="axis"), fod=fod, orientation_parameter="axis")),
        Component("extra", compartment=FODConvolved(response=Zeppelin(parallel_diffusivity="d_in", perpendicular_diffusivity="d_perp", orientation="axis"), fod=fod, orientation_parameter="axis")),
        Component("soma", compartment=Sphere(radius="r_soma", diffusivity="d_soma")),
        Component("csf", compartment=Ball(diffusivity="d_csf")),
    ])
    ps = ParameterSpace([
        Parameter("S0", 1.0, (0.05,2.0)),
        Parameter("d_in", 2.2, (0.5,3.0), vary=vary_d_in),
        Parameter("d_perp", 0.5, (0.0,2.5), vary=vary_d_perp),
        Parameter("d_soma", 3.0, (1.0,3.5), vary=vary_d_soma),
        Parameter("r_soma", 7.0, (1.0,15.0), vary=vary_r_soma),
        Parameter("d_csf", 3.0, (1.5,3.5), vary=vary_d_csf),
        Parameter("kappa", 2.0, (0.0,64.0), vary=vary_kappa),
        Parameter("theta", theta, (0.0,np.pi), vary=False),
        Parameter("phi", phi, (-np.pi,np.pi), vary=False),
    ], simplexes=[Simplex(("neurite_fraction","extra_fraction","soma_fraction","csf_fraction"),(0.50,0.25,0.10,0.15))])
    return ModelPreset(model=model, parameters=ps)

def full_noddi_soma_tort1_model(*, theta=1.0, phi=1.2, vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_csf=False, fod_quadrature_theta=20, fod_quadrature_phi=40):
    fod = WatsonFOD(kappa="kappa", theta="theta", phi="phi", n_theta=fod_quadrature_theta, n_phi=fod_quadrature_phi)
    model = Model([
        Component("neurite", compartment=FODConvolved(response=Stick(diffusivity="d_in", orientation="axis"), fod=fod, orientation_parameter="axis")),
        Component("extra", compartment=FODConvolved(response=NODDIExtraNeurite(parallel_diffusivity="d_in", orientation="axis", neurite_fraction="neurite_fraction", csf_fraction="csf_fraction"), fod=fod, orientation_parameter="axis")),
        Component("soma", compartment=Sphere(radius="r_soma", diffusivity="d_soma")),
        Component("csf", compartment=Ball(diffusivity="d_csf")),
    ])
    ps = ParameterSpace([
        Parameter("S0", 1.0, (0.05,2.0)),
        Parameter("d_in", 2.2, (0.5,3.0), vary=vary_d_in),
        Parameter("d_soma", 3.0, (1.0,3.5), vary=vary_d_soma),
        Parameter("r_soma", 7.0, (1.0,15.0), vary=vary_r_soma),
        Parameter("d_csf", 3.0, (1.5,3.5), vary=vary_d_csf),
        Parameter("kappa", 2.0, (0.0,64.0), vary=vary_kappa),
        Parameter("theta", theta, (0.0,np.pi), vary=False),
        Parameter("phi", phi, (-np.pi,np.pi), vary=False),
    ], simplexes=[Simplex(("neurite_fraction","extra_fraction","soma_fraction","csf_fraction"),(0.50,0.25,0.10,0.15))])
    return ModelPreset(model=model, parameters=ps)

def full_noddi_soma_tort2_model(*, theta=1.0, phi=1.2, vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_csf=False, fod_quadrature_theta=20, fod_quadrature_phi=40):
    fod = WatsonFOD(kappa="kappa", theta="theta", phi="phi", n_theta=fod_quadrature_theta, n_phi=fod_quadrature_phi)
    model = Model([
        Component("neurite", compartment=FODConvolved(response=Stick(diffusivity="d_in", orientation="axis"), fod=fod, orientation_parameter="axis")),
        Component("extra", compartment=FODConvolved(response=NODDISomaExtraNeurite(parallel_diffusivity="d_in", orientation="axis", neurite_fraction="neurite_fraction", soma_fraction="soma_fraction", csf_fraction="csf_fraction"), fod=fod, orientation_parameter="axis")),
        Component("soma", compartment=Sphere(radius="r_soma", diffusivity="d_soma")),
        Component("csf", compartment=Ball(diffusivity="d_csf")),
    ])
    ps = ParameterSpace([
        Parameter("S0", 1.0, (0.05,2.0)),
        Parameter("d_in", 2.2, (0.5,3.0), vary=vary_d_in),
        Parameter("d_soma", 3.0, (1.0,3.5), vary=vary_d_soma),
        Parameter("r_soma", 7.0, (1.0,15.0), vary=vary_r_soma),
        Parameter("d_csf", 3.0, (1.5,3.5), vary=vary_d_csf),
        Parameter("kappa", 2.0, (0.0,64.0), vary=vary_kappa),
        Parameter("theta", theta, (0.0,np.pi), vary=False),
        Parameter("phi", phi, (-np.pi,np.pi), vary=False),
    ], simplexes=[Simplex(("neurite_fraction","extra_fraction","soma_fraction","csf_fraction"),(0.50,0.25,0.10,0.15))])
    return ModelPreset(model=model, parameters=ps)

def full_noddi_soma_tort3_model(*, theta=1.0, phi=1.2, vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_csf=False, fod_quadrature_theta=20, fod_quadrature_phi=40):
    fod = WatsonFOD(kappa="kappa", theta="theta", phi="phi", n_theta=fod_quadrature_theta, n_phi=fod_quadrature_phi)
    model = Model([
        Component("neurite", compartment=FODConvolved(response=Stick(diffusivity="d_in", orientation="axis"), fod=fod, orientation_parameter="axis")),
        Component("extra", compartment=FODConvolved(response=NODDISomaDispersionExtraNeurite(parallel_diffusivity="d_in", orientation="axis", neurite_fraction="neurite_fraction", soma_fraction="soma_fraction", csf_fraction="csf_fraction", kappa="kappa"), fod=fod, orientation_parameter="axis")),
        Component("soma", compartment=Sphere(radius="r_soma", diffusivity="d_soma")),
        Component("csf", compartment=Ball(diffusivity="d_csf")),
    ])
    ps = ParameterSpace([
        Parameter("S0", 1.0, (0.05,2.0)),
        Parameter("d_in", 2.2, (0.5,3.0), vary=vary_d_in),
        Parameter("d_soma", 3.0, (1.0,3.5), vary=vary_d_soma),
        Parameter("r_soma", 7.0, (1.0,15.0), vary=vary_r_soma),
        Parameter("d_csf", 3.0, (1.5,3.5), vary=vary_d_csf),
        Parameter("kappa", 2.0, (0.0,64.0), vary=vary_kappa),
        Parameter("theta", theta, (0.0,np.pi), vary=False),
        Parameter("phi", phi, (-np.pi,np.pi), vary=False),
    ], simplexes=[Simplex(("neurite_fraction","extra_fraction","soma_fraction","csf_fraction"),(0.50,0.25,0.10,0.15))])
    return ModelPreset(model=model, parameters=ps)

def full_no_extra_model(*, theta=1.0, phi=1.2, vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_csf=False, fod_quadrature_theta=20, fod_quadrature_phi=40):
    fod = WatsonFOD(kappa="kappa", theta="theta", phi="phi", n_theta=fod_quadrature_theta, n_phi=fod_quadrature_phi)
    model = Model([
        Component("neurite", compartment=FODConvolved(response=Stick(diffusivity="d_in", orientation="axis"), fod=fod, orientation_parameter="axis")),
        Component("soma", compartment=Sphere(radius="r_soma", diffusivity="d_soma")),
        Component("csf", compartment=Ball(diffusivity="d_csf")),
    ])
    ps = ParameterSpace([
        Parameter("S0", 1.0, (0.05,2.0)),
        Parameter("d_in", 2.2, (0.5,3.0), vary=vary_d_in),
        Parameter("d_soma", 3.0, (1.0,3.5), vary=vary_d_soma),
        Parameter("r_soma", 7.0, (1.0,15.0), vary=vary_r_soma),
        Parameter("d_csf", 3.0, (1.5,3.5), vary=vary_d_csf),
        Parameter("kappa", 2.0, (0.0,64.0), vary=vary_kappa),
        Parameter("theta", theta, (0.0,np.pi), vary=False),
        Parameter("phi", phi, (-np.pi,np.pi), vary=False),
    ], simplexes=[Simplex(("neurite_fraction","soma_fraction","csf_fraction"),(0.75,0.10,0.15))])
    return ModelPreset(model=model, parameters=ps)

def full_no_neurite_model(*, theta=1.0, phi=1.2, vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_ex_parallel=False, vary_d_ex_perp=False, vary_d_csf=False, fod_quadrature_theta=20, fod_quadrature_phi=40):
    fod = WatsonFOD(kappa="kappa", theta="theta", phi="phi", n_theta=fod_quadrature_theta, n_phi=fod_quadrature_phi)
    model = Model([
        Component("extra", compartment=FODConvolved(response=Zeppelin(parallel_diffusivity="d_ex_parallel", perpendicular_diffusivity="d_ex_perp", orientation="axis"), fod=fod, orientation_parameter="axis")),
        Component("soma", compartment=Sphere(radius="r_soma", diffusivity="d_soma")),
        Component("csf", compartment=Ball(diffusivity="d_csf")),
    ])
    ps = ParameterSpace([
        Parameter("S0", 1.0, (0.05,2.0)),
        Parameter("d_ex_parallel", 2.2, (0.5,3.5), vary=vary_d_ex_parallel),
        Parameter("d_ex_perp", 0.5, (0.0,2.5), vary=vary_d_ex_perp),
        Parameter("d_soma", 3.0, (1.0,3.5), vary=vary_d_soma),
        Parameter("r_soma", 7.0, (1.0,15.0), vary=vary_r_soma),
        Parameter("d_csf", 3.0, (1.5,3.5), vary=vary_d_csf),
        Parameter("kappa", 2.0, (0.0,64.0), vary=vary_kappa),
        Parameter("theta", theta, (0.0,np.pi), vary=False),
        Parameter("phi", phi, (-np.pi,np.pi), vary=False),
    ], simplexes=[Simplex(("extra_fraction","soma_fraction","csf_fraction"),(0.75,0.10,0.15))])
    return ModelPreset(model=model, parameters=ps)

def full_no_fw_model(*, theta=1.0, phi=1.2, vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_perp=False, fod_quadrature_theta=20, fod_quadrature_phi=40):
    """Full NODDI-style model with the explicit free-water compartment removed.

    Contains neurite + extra-axonal + soma. The synthetic truth still contains
    free water, so this is a deliberate model-misspecification/robustness test.
    """
    fod = WatsonFOD(kappa="kappa", theta="theta", phi="phi", n_theta=fod_quadrature_theta, n_phi=fod_quadrature_phi)
    model = Model([
        Component("neurite", compartment=FODConvolved(response=Stick(diffusivity="d_in", orientation="axis"), fod=fod, orientation_parameter="axis")),
        Component("extra", compartment=FODConvolved(response=Zeppelin(parallel_diffusivity="d_in", perpendicular_diffusivity="d_perp", orientation="axis"), fod=fod, orientation_parameter="axis")),
        Component("soma", compartment=Sphere(radius="r_soma", diffusivity="d_soma")),
    ])
    ps = ParameterSpace([
        Parameter("S0", 1.0, (0.05,2.0)),
        Parameter("d_in", 2.2, (0.5,3.0), vary=vary_d_in),
        Parameter("d_perp", 0.5, (0.0,2.5), vary=vary_d_perp),
        Parameter("d_soma", 3.0, (1.0,3.5), vary=vary_d_soma),
        Parameter("r_soma", 7.0, (1.0,15.0), vary=vary_r_soma),
        Parameter("kappa", 2.0, (0.0,64.0), vary=vary_kappa),
        Parameter("theta", theta, (0.0,np.pi), vary=False),
        Parameter("phi", phi, (-np.pi,np.pi), vary=False),
    ], simplexes=[Simplex(("neurite_fraction","extra_fraction","soma_fraction"),(0.625,0.125,0.25))])
    return ModelPreset(model=model, parameters=ps)


## 2. Parameter-freedom configurations

Every configuration keeps the relevant fractions and `r_soma` free. Nuisance/mechanism freedom is then added progressively. The no-extra family has the same progression but without `d_perp`, because it has no extra-axonal compartment.

In [96]:
MODEL_CONFIGS = {
    # Original unconstrained Zeppelin model
    # "full_core": {"family": "full_noddi_soma", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_perp=False, vary_d_csf=False)},
    # "full_orientation_free": {"family": "full_noddi_soma", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=False, vary_d_in=False, vary_d_perp=False, vary_d_csf=False)},
    "full_soma_diffusivity_free": {"family": "full_noddi_soma", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=True, vary_d_in=False, vary_d_perp=False, vary_d_csf=False)},
    # "full_all_microstructure_free": {"family": "full_noddi_soma", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_in=True, vary_d_perp=True, vary_d_csf=False)},
    # Tortuosity 1: d_ex_perp = d_in * (1 - v_ic)
    # "full_tort1_core": {"family": "full_noddi_soma_tort1", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_csf=False)},
    # "full_tort1_orientation_free": {"family": "full_noddi_soma_tort1", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=False, vary_d_in=False, vary_d_csf=False)},
    "full_tort1_soma_diffusivity_free": {"family": "full_noddi_soma_tort1", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=True, vary_d_in=False, vary_d_csf=False)},
    # "full_tort1_all_microstructure_free": {"family": "full_noddi_soma_tort1", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_in=True, vary_d_csf=False)},
    # Tortuosity 2: soma-aware d_ex_parallel and d_ex_perp
    # "full_tort2_core": {"family": "full_noddi_soma_tort2", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_csf=False)},
    # "full_tort2_orientation_free": {"family": "full_noddi_soma_tort2", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=False, vary_d_in=False, vary_d_csf=False)},
    "full_tort2_soma_diffusivity_free": {"family": "full_noddi_soma_tort2", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=True, vary_d_in=False, vary_d_csf=False)},
    # "full_tort2_all_microstructure_free": {"family": "full_noddi_soma_tort2", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_in=True, vary_d_csf=False)},
    # Tortuosity 3: soma-aware + ODI-dependent dispersion coupling
    # "full_tort3_core": {"family": "full_noddi_soma_tort3", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_csf=False)},
    # "full_tort3_orientation_free": {"family": "full_noddi_soma_tort3", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=False, vary_d_in=False, vary_d_csf=False)},
    "full_tort3_soma_diffusivity_free": {"family": "full_noddi_soma_tort3", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=True, vary_d_in=False, vary_d_csf=False)},
    # "full_tort3_all_microstructure_free": {"family": "full_noddi_soma_tort3", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_in=True, vary_d_csf=False)},
    # Noddi with soma but no extra-neurite compartment
    # "no_extra_core": {"family": "full_no_extra", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_csf=False)},
    # "no_extra_orientation_free": {"family": "full_no_extra", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=False, vary_d_in=False, vary_d_csf=False)},
    # "no_extra_soma_diffusivity_free": {"family": "full_no_extra", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_in=False, vary_d_csf=False)},
    # "no_extra_all_microstructure_free": {"family": "full_no_extra", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_in=True, vary_d_csf=False)},
     # Noddi with soma but no neurite compartment
    # "no_neurite_core": {"family": "full_no_neurite", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_ex_parallel=False, vary_d_ex_perp=False, vary_d_csf=False)},
    # "no_neurite_orientation_free": {"family": "full_no_neurite", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=False, vary_d_ex_parallel=False, vary_d_ex_perp=False, vary_d_csf=False)},
    # "no_neurite_soma_diffusivity_free": {"family": "full_no_neurite", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_ex_parallel=False, vary_d_ex_perp=False, vary_d_csf=False)},
    # "no_neurite_all_microstructure_free": {"family": "full_no_neurite", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_ex_parallel=True, vary_d_ex_perp=True, vary_d_csf=False)},
    # Noddi with soma but no free water compartment
    # "no_fw_core": {"family": "full_no_fw", "kwargs": dict(vary_r_soma=True, vary_kappa=False, vary_d_soma=False, vary_d_in=False, vary_d_perp=False)},
    # "no_fw_orientation_free": {"family": "full_no_fw", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=False, vary_d_in=False, vary_d_perp=False)},
    # "no_fw_soma_diffusivity_free": {"family": "full_no_fw", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_in=False, vary_d_perp=False)},
    # "no_fw_all_microstructure_free": {"family": "full_no_fw", "kwargs": dict(vary_r_soma=True, vary_kappa=True, vary_d_soma=True, vary_d_in=True, vary_d_perp=False)},
}

def build_model_family(family, kwargs):
    if family == "full_noddi_soma": return full_noddi_soma_model(**kwargs)
    if family == "full_noddi_soma_tort1": return full_noddi_soma_tort1_model(**kwargs)
    if family == "full_noddi_soma_tort2": return full_noddi_soma_tort2_model(**kwargs)
    if family == "full_noddi_soma_tort3": return full_noddi_soma_tort3_model(**kwargs)
    if family == "full_no_extra": return full_no_extra_model(**kwargs)
    if family == "full_no_neurite": return full_no_neurite_model(**kwargs)
    if family == "full_no_fw": return full_no_fw_model(**kwargs)
    raise ValueError(f"Unknown family: {family}")

models = {name: build_model_family(spec["family"], spec["kwargs"]) for name,spec in MODEL_CONFIGS.items()}
for name,preset in models.items():
    print(f"\n{name}\n  family: {MODEL_CONFIGS[name]['family']}\n  free: {preset.parameters.varying_names}")



full_soma_diffusivity_free
  family: full_noddi_soma
  free: ('S0', 'd_soma', 'r_soma', 'neurite_fraction/csf_fraction', 'extra_fraction/csf_fraction', 'soma_fraction/csf_fraction')

full_tort1_soma_diffusivity_free
  family: full_noddi_soma_tort1
  free: ('S0', 'd_soma', 'r_soma', 'neurite_fraction/csf_fraction', 'extra_fraction/csf_fraction', 'soma_fraction/csf_fraction')

full_tort2_soma_diffusivity_free
  family: full_noddi_soma_tort2
  free: ('S0', 'd_soma', 'r_soma', 'neurite_fraction/csf_fraction', 'extra_fraction/csf_fraction', 'soma_fraction/csf_fraction')

full_tort3_soma_diffusivity_free
  family: full_noddi_soma_tort3
  free: ('S0', 'd_soma', 'r_soma', 'neurite_fraction/csf_fraction', 'extra_fraction/csf_fraction', 'soma_fraction/csf_fraction')


In [97]:
TRUE_BASE = {
    "S0": 1.0,
    "d_in": 2.2,
    "d_perp": 0.5,
    "d_soma": 3.0,
    "d_csf": 3.0,
    "kappa": 2.0,
    "theta": 1.0,
    "phi": 1.2,
}

TRUTH_SCENARIOS = {
    "truth_1": {
        "label": "f_soma=0.10, r_soma=5, f_fw=0.15",
        "soma_fraction": 0.10,
        "r_soma": 5.0,
        "csf_fraction": 0.15,
    },
    "truth_2": {
        "label": "f_soma=0.15, r_soma=7, f_fw=0.20",
        "soma_fraction": 0.15,
        "r_soma": 7.0,
        "csf_fraction": 0.20,
    },
    "truth_3": {
        "label": "f_soma=0.20, r_soma=8, f_fw=0.25",
        "soma_fraction": 0.20,
        "r_soma": 8.0,
        "csf_fraction": 0.25,
    },
}

def make_full_truth(scenario, neurite_to_extra=2.0):
    f_soma = scenario["soma_fraction"]
    f_fw = scenario["csf_fraction"]

    remaining = 1.0 - f_soma - f_fw

    extra_fraction = remaining / (1.0 + neurite_to_extra)
    neurite_fraction = neurite_to_extra * extra_fraction

    truth = dict(TRUE_BASE)
    truth.update({
        "r_soma": scenario["r_soma"],
        "neurite_fraction": neurite_fraction,
        "extra_fraction": extra_fraction,
        "soma_fraction": f_soma,
        "csf_fraction": f_fw,
    })

    return truth

TRUTH_BY_SCENARIO = {
    name: make_full_truth(scenario)
    for name, scenario in TRUTH_SCENARIOS.items()
}

truth_df = pd.DataFrame([
    {
        "scenario": name,
        "r_soma": truth["r_soma"],
        "f_soma": truth["soma_fraction"],
        "f_fw": truth["csf_fraction"],
        "f_neurite": truth["neurite_fraction"],
        "f_extra": truth["extra_fraction"],
    }
    for name, truth in TRUTH_BY_SCENARIO.items()
])

display(truth_df)

,scenario,r_soma,f_soma,f_fw,f_neurite,f_extra
0,truth_1,5.0,0.10,0.15,0.500000,0.250000
1,truth_2,7.0,0.15,0.20,0.433333,0.216667
2,truth_3,8.0,0.20,0.25,0.366667,0.183333


## 3. Simulation / fitting adapter

In [98]:
def simulate_signal(preset, acquisition, parameters):
    model=preset.model
    for name in ("predict","evaluate","signal"):
        method=getattr(model,name,None)
        if method is not None:
            for kwargs in ({"acquisition":acquisition,"parameters":parameters},{"parameters":parameters,"acquisition":acquisition}):
                try: return np.asarray(method(**kwargs),dtype=float)
                except TypeError: pass
    if callable(model):
        for args in ((acquisition,parameters),(parameters,acquisition)):
            try: return np.asarray(model(*args),dtype=float)
            except TypeError: pass
    raise RuntimeError("Could not identify signal-evaluation API.")

def add_gaussian_noise(signal,snr,seed):
    signal=np.asarray(signal,dtype=float); rng=np.random.default_rng(seed)
    sigma=float(signal.max())/float(snr)
    return signal+rng.normal(0.0,sigma,size=signal.shape)

def _normalise_fit_result(result):
    out=dict(result) if isinstance(result,dict) else {}
    if not out:
        for attr in ("physical_parameters","parameters","fit_parameters"):
            if hasattr(result,attr):
                try: out.update(dict(getattr(result,attr))); break
                except Exception: pass
        for key in ("rmse","success","n_evaluations","residuals"):
            if hasattr(result,key): out[key]=getattr(result,key)
    if "rmse" not in out and "residuals" in out:
        r=np.asarray(out["residuals"],dtype=float); out["rmse"]=float(np.sqrt(np.mean(r*r)))
    return out

def fit_one(preset,acquisition,data,n_starts=5,max_nfev=300,seed=0):
    fitter=LeastSquaresFitter(preset.model,preset.parameters)
    attempts=[
        lambda: fitter.fit(acquisition,data,n_starts=n_starts,max_nfev=max_nfev,seed=seed),
        lambda: fitter.fit(acquisition,data,n_starts=n_starts,max_nfev=max_nfev),
        lambda: fitter.fit(acquisition,data),
    ]
    last=None
    for attempt in attempts:
        try: return _normalise_fit_result(attempt())
        except (TypeError,AttributeError) as exc: last=exc
    raise RuntimeError("Could not identify LeastSquaresFitter API") from last


## 4. Sanity check

The test below fits one common truth dataset with every model family. Pay particular attention to the new `no_extra_*` rows.

In [99]:
#TEST_SCENARIO="truth_2"
#test_truth=TRUTH_BY_SCENARIO[TEST_SCENARIO]
#truth_model=full_noddi_soma_model(vary_r_soma=False,vary_kappa=False,vary_d_soma=False,vary_d_in=False,vary_d_perp=False,vary_d_csf=False)
#signal_true_test=simulate_signal(truth_model,acquisition,test_truth)
#data_test=add_gaussian_noise(signal_true_test,SNR,BASE_SEED)
#rows=[]
#for model_name,preset in models.items():
#    fit=fit_one(preset,acquisition,data_test,N_STARTS,MAX_NFEV,BASE_SEED)
#    rows.append({"model":model_name,"success":fit.get("success",True),"rmse":fit.get("rmse",np.nan),"r_soma":fit.get("r_soma",np.nan),"soma_fraction":fit.get("soma_fraction",np.nan),"csf_fraction":fit.get("csf_fraction",np.nan),"kappa":fit.get("kappa",np.nan)})
#display(pd.DataFrame(rows))


## 5. Main paired benchmark

For every scenario and repeat, the same noisy dataset is passed to every model variant. The truth is always generated by the full NODDI+soma model.

## Additional model family: neurite + extra + soma (no explicit free water)

This ablation removes the explicit CSF/free-water compartment while the synthetic truth still contains free water. The no-FW models therefore cannot recover `f_fw` directly; instead, increasing true `f_fw` across the three scenarios is used to test whether unmodelled free-water signal biases the parameters we care about, especially `r_soma` and `f_soma`.

The four no-FW configurations mirror the existing freedom progression: core, `kappa` free, `d_soma` free, and all WM/soma microstructure free.

In [100]:
RESULTS_FILE = Path("results_checkpoint.csv")

results = []

for scenario_name, truth in TRUTH_BY_SCENARIO.items():
    print(f"\nScenario: {scenario_name} - {TRUTH_SCENARIOS[scenario_name]['label']}")
    truth_signal = simulate_signal(truth_model, acquisition, truth)

    for rep in range(N_REPEATS):
        seed = BASE_SEED + rep

        print(
            f"\n  Repeat {rep + 1}/{N_REPEATS}",
            flush=True,
        )

        data_noisy = add_gaussian_noise(truth_signal, SNR, seed)

        for i, (model_name, preset) in enumerate(models.items(), 1):
            print(f"    {i}/{len(models)}: {model_name}", flush=True)

            try:
                fit = fit_one(
                    preset,
                    acquisition,
                    data_noisy,
                    N_STARTS,
                    MAX_NFEV,
                    seed,
                )

                row = {
                    "scenario": scenario_name,
                    "model": model_name,
                    "family": MODEL_CONFIGS[model_name]["family"],
                    "repeat": rep,
                    "seed": seed,
                    "success": bool(fit.get("success", True)),
                    "rmse": fit.get("rmse", np.nan),
                    "n_evaluations": fit.get("n_evaluations", np.nan),
                    "true_r_soma": truth["r_soma"],
                    "true_soma_fraction": truth["soma_fraction"],
                    "true_f_fw": truth["csf_fraction"],
                }

                for name in [
                    "S0",
                    "r_soma",
                    "soma_fraction",
                    "csf_fraction",
                    "neurite_fraction",
                    "extra_fraction",
                    "kappa",
                    "d_in",
                    "d_perp",
                    "d_soma",
                    "d_csf",
                    "d_ex_parallel",
                    "d_ex_perp",
                ]:
                    if name in fit:
                        row[name] = fit[name]

            except Exception as exc:
                row = {
                    "scenario": scenario_name,
                    "model": model_name,
                    "family": MODEL_CONFIGS[model_name]["family"],
                    "repeat": rep,
                    "seed": seed,
                    "success": False,
                    "rmse": np.nan,
                    "n_evaluations": np.nan,
                    "error": repr(exc),
                    "true_r_soma": truth["r_soma"],
                    "true_soma_fraction": truth["soma_fraction"],
                    "true_f_fw": truth["csf_fraction"],
                }

                print(f"    FAILED: {model_name}: {exc}")

            results.append(row)

            # Save after every fit
            pd.DataFrame(results).to_csv(
                RESULTS_FILE,
                index=False,
            )

results_df = pd.DataFrame(results)

print("Rows:", len(results_df))
display(results_df.head())


Scenario: truth_1 - f_soma=0.10, r_soma=5, f_fw=0.15

  Repeat 1/1
    1/4: full_soma_diffusivity_free
    2/4: full_tort1_soma_diffusivity_free
    3/4: full_tort2_soma_diffusivity_free
    4/4: full_tort3_soma_diffusivity_free

Scenario: truth_2 - f_soma=0.15, r_soma=7, f_fw=0.20

  Repeat 1/1
    1/4: full_soma_diffusivity_free
    2/4: full_tort1_soma_diffusivity_free
    3/4: full_tort2_soma_diffusivity_free
    4/4: full_tort3_soma_diffusivity_free

Scenario: truth_3 - f_soma=0.20, r_soma=8, f_fw=0.25

  Repeat 1/1
    1/4: full_soma_diffusivity_free
    2/4: full_tort1_soma_diffusivity_free
    3/4: full_tort2_soma_diffusivity_free
    4/4: full_tort3_soma_diffusivity_free
Rows: 12


,scenario,model,family,repeat,seed,success,rmse,n_evaluations,true_r_soma,true_soma_fraction,true_f_fw,S0,r_soma,soma_fraction,csf_fraction,neurite_fraction,extra_fraction,kappa,d_in,d_perp,d_soma,d_csf
0,truth_1,full_soma_diffusivity_free,full_noddi_soma,0,1000,True,0.009293,22,5.0,0.10,0.15,0.999571,4.474398,0.082227,0.127882,0.476029,0.313861,2.0,2.2,0.5,3.228306,3.0
1,truth_1,full_tort1_soma_diffusivity_free,full_noddi_soma_tort1,0,1000,True,0.009394,24,5.0,0.10,0.15,0.999231,4.874432,0.116350,0.106992,0.538165,0.238493,2.0,2.2,NaN,1.426783,3.0
2,truth_1,full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,0,1000,True,0.009380,47,5.0,0.10,0.15,0.999269,4.712124,0.099749,0.123028,0.532152,0.245071,2.0,2.2,NaN,1.802549,3.0
3,truth_1,full_tort3_soma_diffusivity_free,full_noddi_soma_tort3,0,1000,True,0.009537,30,5.0,0.10,0.15,0.999247,4.687424,0.099495,0.128242,0.545803,0.226460,2.0,2.2,NaN,1.032474,3.0
4,truth_2,full_soma_diffusivity_free,full_noddi_soma,0,1000,True,0.009347,22,7.0,0.15,0.20,0.999594,8.053829,0.180136,0.189137,0.448104,0.182622,2.0,2.2,0.5,1.575722,3.0


## 6. Primary recovery metrics

In [101]:
PRIMARY_PARAMS={"r_soma":"true_r_soma","soma_fraction":"true_soma_fraction","csf_fraction":"true_f_fw"}
metric_rows=[]
for (scenario,model_name),sub in results_df.groupby(["scenario","model"]):
    for p,truth_col in PRIMARY_PARAMS.items():
        if p not in sub.columns: continue
        vals=pd.to_numeric(sub[p],errors="coerce"); truth=float(sub[truth_col].iloc[0]); err=vals-truth
        metric_rows.append({"scenario":scenario,"model":model_name,"family":MODEL_CONFIGS[model_name]["family"],"parameter":p,"truth":truth,"mean":vals.mean(),"bias":err.mean(),"mae":err.abs().mean(),"rmse":float(np.sqrt(np.mean(err**2))),"sd":vals.std(ddof=1),"relative_rmse":float(np.sqrt(np.mean(err**2))/abs(truth)) if truth else np.nan,"n_valid":int(vals.notna().sum()),"n_total":len(vals)})
metrics_df=pd.DataFrame(metric_rows)
robustness=(results_df.groupby(["scenario","model"]).agg(success_rate=("success","mean"),rmse_median=("rmse","median"),rmse_mean=("rmse","mean"),n_evaluations_median=("n_evaluations","median")).reset_index())
display(metrics_df.sort_values(["scenario","parameter","rmse"]))
display(robustness)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(results_df)


,scenario,model,family,parameter,truth,mean,bias,mae,rmse,sd,relative_rmse,n_valid,n_total
11,truth_1,full_tort3_soma_diffusivity_free,full_noddi_soma_tort3,csf_fraction,0.15,0.128242,-0.021758,0.021758,0.021758,NaN,0.145051,1,1
2,truth_1,full_soma_diffusivity_free,full_noddi_soma,csf_fraction,0.15,0.127882,-0.022118,0.022118,0.022118,NaN,0.147452,1,1
8,truth_1,full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,csf_fraction,0.15,0.123028,-0.026972,0.026972,0.026972,NaN,0.179815,1,1
5,truth_1,full_tort1_soma_diffusivity_free,full_noddi_soma_tort1,csf_fraction,0.15,0.106992,-0.043008,0.043008,0.043008,NaN,0.286717,1,1
3,truth_1,full_tort1_soma_diffusivity_free,full_noddi_soma_tort1,r_soma,5.00,4.874432,-0.125568,0.125568,0.125568,NaN,0.025114,1,1
6,truth_1,full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,r_soma,5.00,4.712124,-0.287876,0.287876,0.287876,NaN,0.057575,1,1
9,truth_1,full_tort3_soma_diffusivity_free,full_noddi_soma_tort3,r_soma,5.00,4.687424,-0.312576,0.312576,0.312576,NaN,0.062515,1,1
0,truth_1,full_soma_diffusivity_free,full_noddi_soma,r_soma,5.00,4.474398,-0.525602,0.525602,0.525602,NaN,0.105120,1,1
7,truth_1,full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,soma_fraction,0.10,0.099749,-0.000251,0.000251,0.000251,NaN,0.002507,1,1
10,truth_1,full_tort3_soma_diffusivity_free,full_noddi_soma_tort3,soma_fraction,0.10,0.099495,-0.000505,0.000505,0.000505,NaN,0.005054,1,1


,scenario,model,success_rate,rmse_median,rmse_mean,n_evaluations_median
0,truth_1,full_soma_diffusivity_free,1.0,0.009293,0.009293,22.0
1,truth_1,full_tort1_soma_diffusivity_free,1.0,0.009394,0.009394,24.0
2,truth_1,full_tort2_soma_diffusivity_free,1.0,0.009380,0.009380,47.0
3,truth_1,full_tort3_soma_diffusivity_free,1.0,0.009537,0.009537,30.0
4,truth_2,full_soma_diffusivity_free,1.0,0.009347,0.009347,22.0
5,truth_2,full_tort1_soma_diffusivity_free,1.0,0.009491,0.009491,25.0
6,truth_2,full_tort2_soma_diffusivity_free,1.0,0.009391,0.009391,29.0
7,truth_2,full_tort3_soma_diffusivity_free,1.0,0.009557,0.009557,25.0
8,truth_3,full_soma_diffusivity_free,1.0,0.009313,0.009313,23.0
9,truth_3,full_tort1_soma_diffusivity_free,1.0,0.009473,0.009473,30.0


,scenario,model,family,repeat,seed,success,rmse,n_evaluations,true_r_soma,true_soma_fraction,true_f_fw,S0,r_soma,soma_fraction,csf_fraction,neurite_fraction,extra_fraction,kappa,d_in,d_perp,d_soma,d_csf
0,truth_1,full_soma_diffusivity_free,full_noddi_soma,0,1000,True,0.009293,22,5.0,0.10,0.15,0.999571,4.474398,0.082227,0.127882,0.476029,0.313861,2.0,2.2,0.5,3.228306,3.0
1,truth_1,full_tort1_soma_diffusivity_free,full_noddi_soma_tort1,0,1000,True,0.009394,24,5.0,0.10,0.15,0.999231,4.874432,0.116350,0.106992,0.538165,0.238493,2.0,2.2,NaN,1.426783,3.0
2,truth_1,full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,0,1000,True,0.009380,47,5.0,0.10,0.15,0.999269,4.712124,0.099749,0.123028,0.532152,0.245071,2.0,2.2,NaN,1.802549,3.0
3,truth_1,full_tort3_soma_diffusivity_free,full_noddi_soma_tort3,0,1000,True,0.009537,30,5.0,0.10,0.15,0.999247,4.687424,0.099495,0.128242,0.545803,0.226460,2.0,2.2,NaN,1.032474,3.0
4,truth_2,full_soma_diffusivity_free,full_noddi_soma,0,1000,True,0.009347,22,7.0,0.15,0.20,0.999594,8.053829,0.180136,0.189137,0.448104,0.182622,2.0,2.2,0.5,1.575722,3.0
5,truth_2,full_tort1_soma_diffusivity_free,full_noddi_soma_tort1,0,1000,True,0.009491,25,7.0,0.15,0.20,0.999064,7.081789,0.175173,0.142493,0.463161,0.219173,2.0,2.2,NaN,2.977116,3.0
6,truth_2,full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,0,1000,True,0.009391,29,7.0,0.15,0.20,0.999187,6.996962,0.133523,0.166292,0.458088,0.242097,2.0,2.2,NaN,1.427946,3.0
7,truth_2,full_tort3_soma_diffusivity_free,full_noddi_soma_tort3,0,1000,True,0.009557,25,7.0,0.15,0.20,0.999131,8.112095,0.138441,0.169256,0.466039,0.226263,2.0,2.2,NaN,1.096421,3.0
8,truth_3,full_soma_diffusivity_free,full_noddi_soma,0,1000,True,0.009313,23,8.0,0.20,0.25,0.999460,8.543855,0.236213,0.229926,0.377181,0.156681,2.0,2.2,0.5,2.855469,3.0
9,truth_3,full_tort1_soma_diffusivity_free,full_noddi_soma_tort1,0,1000,True,0.009473,30,8.0,0.20,0.25,0.999072,7.844304,0.202367,0.176818,0.388751,0.232064,2.0,2.2,NaN,1.954459,3.0


## 7. Primary recovery plots

In [104]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================================
# 1. Rebuild recovery metrics from the CURRENT results_df
# ============================================================================

PRIMARY_PARAMS = {
    "r_soma": "true_r_soma",
    "soma_fraction": "true_soma_fraction",
    "csf_fraction": "true_f_fw",
}

metric_rows = []

for (scenario, model_name), sub in results_df.groupby(
    ["scenario", "model"]
):

    for parameter, truth_column in PRIMARY_PARAMS.items():

        # Model may not contain this parameter.
        if parameter not in sub.columns:
            continue

        values = pd.to_numeric(
            sub[parameter],
            errors="coerce",
        )

        truth = float(sub[truth_column].iloc[0])

        valid = values.notna()

        if valid.sum() == 0:
            continue

        values_valid = values[valid]
        errors = values_valid - truth

        metric_rows.append({
            "scenario": scenario,
            "model": model_name,
            "family": MODEL_CONFIGS[model_name]["family"],
            "parameter": parameter,
            "truth": truth,
            "mean": values_valid.mean(),
            "bias": errors.mean(),
            "mae": errors.abs().mean(),
            "rmse": np.sqrt(np.mean(errors**2)),
            "sd": values_valid.std(ddof=1),
            "relative_rmse": (
                np.sqrt(np.mean(errors**2)) / abs(truth)
                if truth != 0
                else np.nan
            ),
            "n_valid": int(valid.sum()),
            "n_total": len(values),
        })

metrics_df = pd.DataFrame(metric_rows)

print("Parameters available in metrics_df:")
print(metrics_df["parameter"].unique())

print("\nNumber of valid results:")
display(
    metrics_df.groupby("parameter")["n_valid"]
    .sum()
    .to_frame()
)


# ============================================================================
# 2. Model labels
# ============================================================================

MODEL_ORDER = list(models.keys())

MODEL_LABELS = {
    # Full model
    "full_core": "Full core",
    "full_orientation_free": "Full + kappa",
    "full_soma_diffusivity_free": "Full + kappa + d_soma",
    "full_all_microstructure_free": "Full all-free",

    # Full + Tortuosity 1
    "full_tort1_core": "Tort1 core",
    "full_tort1_orientation_free": "Tort1 + kappa",
    "full_tort1_soma_diffusivity_free": "Tort1 + kappa + d_soma",
    "full_tort1_all_microstructure_free": "Tort1 all-free",

    # Full + Tortuosity 2
    "full_tort2_core": "Tort2 core",
    "full_tort2_orientation_free": "Tort2 + kappa",
    "full_tort2_soma_diffusivity_free": "Tort2 + kappa + d_soma",
    "full_tort2_all_microstructure_free": "Tort2 all-free",

    # Full + Tortuosity 3
    "full_tort3_core": "Tort3 core",
    "full_tort3_orientation_free": "Tort3 + kappa",
    "full_tort3_soma_diffusivity_free": "Tort3 + kappa + d_soma",
    "full_tort3_all_microstructure_free": "Tort3 all-free",

    # No extra
    "no_extra_core": "No extra core",
    "no_extra_orientation_free": "No extra + kappa",
    "no_extra_soma_diffusivity_free": "No extra + kappa + d_soma",
    "no_extra_all_microstructure_free": "No extra all-free",

    # No neurite
    "no_neurite_core": "No neurite core",
    "no_neurite_orientation_free": "No neurite + kappa",
    "no_neurite_soma_diffusivity_free": "No neurite + kappa + d_soma",
    "no_neurite_all_microstructure_free": "No neurite all-free",

    # No free water
    "no_fw_core": "No FW core",
    "no_fw_orientation_free": "No FW + kappa",
    "no_fw_soma_diffusivity_free": "No FW + kappa + d_soma",
    "no_fw_all_microstructure_free": "No FW all-free",
}


PARAMS = [
    "r_soma",
    "soma_fraction",
    "csf_fraction",
]

LABELS = {
    "r_soma": "r_soma",
    "soma_fraction": "f_soma",
    "csf_fraction": "f_fw",
}


# ============================================================================
# 3. Save the three recovery plots
# ============================================================================

PLOT_DIR = Path("recovery_plots")
PLOT_DIR.mkdir(exist_ok=True)

for parameter in PARAMS:

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(21, 5),
        constrained_layout=True,
    )

    for ax, (scenario_name, scenario) in zip(
        axes,
        TRUTH_SCENARIOS.items(),
    ):

        sub = metrics_df[
            (metrics_df["scenario"] == scenario_name)
            & (metrics_df["parameter"] == parameter)
        ].copy()

        # Put models in the requested order.
        sub = (
            sub.set_index("model")
            .reindex(MODEL_ORDER)
            .dropna(subset=["mean"])
        )

        if sub.empty:
            ax.text(
                0.5,
                0.5,
                "No valid results",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.set_title(
                TRUTH_SCENARIOS[scenario_name]["label"]
            )
            ax.axis("off")
            continue

        x = np.arange(len(sub))

        ax.errorbar(
            x,
            sub["mean"],
            yerr=sub["sd"],
            fmt="o",
            capsize=4,
        )

        # True value
        ax.axhline(
            sub["truth"].iloc[0],
            linestyle="--",
            linewidth=1.5,
        )

        ax.set_xticks(x)

        ax.set_xticklabels(
            [
                MODEL_LABELS.get(model, model)
                for model in sub.index
            ],
            rotation=70,
            ha="right",
        )

        ax.set_ylabel(LABELS[parameter])

        ax.set_title(
            TRUTH_SCENARIOS[scenario_name]["label"]
        )

        ax.grid(
            True,
            alpha=0.25,
        )

    fig.suptitle(
        f"Recovery of {LABELS[parameter]}",
        y=1.02,
    )

    filename = (
        PLOT_DIR
        / f"recovery_{parameter}.png"
    )

    fig.savefig(
        filename,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(fig)

    print(f"Saved: {filename}")


Parameters available in metrics_df:
<StringArray>
['r_soma', 'soma_fraction', 'csf_fraction']
Length: 3, dtype: str

Number of valid results:


,n_valid
parameter,
csf_fraction,12
r_soma,12
soma_fraction,12


Saved: recovery_plots/recovery_r_soma.png
Saved: recovery_plots/recovery_soma_fraction.png
Saved: recovery_plots/recovery_csf_fraction.png


## 8. Overall Model Performance for Recovery of Key Biological Parameters
Mean Relative RMSE Across Truth Scenarios
AND
Best Parameter-Freedom Configuration Within Each Model Family

In [105]:
family_summary=(metrics_df.groupby(["model","family","parameter"])["relative_rmse"].mean().reset_index())
family_pivot=family_summary.pivot_table(index=["model","family"],columns="parameter",values="relative_rmse").reindex(columns=PARAMS)
family_pivot["mean_primary_relative_rmse"]=family_pivot[PARAMS].mean(axis=1)
family_pivot=family_pivot.sort_values("mean_primary_relative_rmse")
display(family_pivot)

print("Best configuration within each family:")
display(family_pivot.reset_index().sort_values(["family","mean_primary_relative_rmse"]).groupby("family",as_index=False).first())


,parameter,r_soma,soma_fraction,csf_fraction,mean_primary_relative_rmse
model,family,,,,
full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,0.040451,0.126734,0.168152,0.111779
full_soma_diffusivity_free,full_noddi_soma,0.107883,0.186569,0.094022,0.129491
full_tort3_soma_diffusivity_free,full_noddi_soma_tort3,0.152034,0.111050,0.150371,0.137819
full_tort1_soma_diffusivity_free,full_noddi_soma_tort1,0.018753,0.114383,0.288993,0.140710


Best configuration within each family:


parameter,family,model,r_soma,soma_fraction,csf_fraction,mean_primary_relative_rmse
0,full_noddi_soma,full_soma_diffusivity_free,0.107883,0.186569,0.094022,0.129491
1,full_noddi_soma_tort1,full_tort1_soma_diffusivity_free,0.018753,0.114383,0.288993,0.140710
2,full_noddi_soma_tort2,full_tort2_soma_diffusivity_free,0.040451,0.126734,0.168152,0.111779
3,full_noddi_soma_tort3,full_tort3_soma_diffusivity_free,0.152034,0.111050,0.150371,0.137819


# Recovery of Key Biological Parameters for the Five Best-Performing Model Configurations

In [106]:
# ============================================================================
# Top 5 models based on soma recovery
# ============================================================================

TOP_N = 5

family_pivot["mean_soma_relative_rmse"] = (
    family_pivot[["r_soma", "soma_fraction"]].mean(axis=1)
)

top_models = (
    family_pivot
    .sort_values("mean_primary_relative_rmse")
    .head(TOP_N)
    .index
    .get_level_values("model")
    .tolist()
)

print("Top 5 models:")
for i, model in enumerate(top_models, 1):
    print(f"{i}. {MODEL_LABELS.get(model, model)}")


# ============================================================================
# Plot recovery for top 5 models
# ============================================================================

TOP_PLOT_DIR = Path("recovery_plots_top5")
TOP_PLOT_DIR.mkdir(exist_ok=True)

for parameter in PARAMS:

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 5),
        constrained_layout=True,
    )

    for ax, (scenario_name, scenario) in zip(
        axes,
        TRUTH_SCENARIOS.items(),
    ):

        sub = metrics_df[
            (metrics_df["scenario"] == scenario_name)
            & (metrics_df["parameter"] == parameter)
            & (metrics_df["model"].isin(top_models))
        ].copy()

        sub = (
            sub.set_index("model")
            .reindex(top_models)
            .dropna(subset=["mean"])
        )

        if sub.empty:
            ax.text(
                0.5, 0.5,
                "No valid results",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.axis("off")
            continue

        x = np.arange(len(sub))

        ax.errorbar(
            x,
            sub["mean"],
            yerr=sub["sd"],
            fmt="o",
            capsize=5,
        )

        # True value
        ax.axhline(
            sub["truth"].iloc[0],
            linestyle="--",
            linewidth=1.5,
        )

        ax.set_xticks(x)

        ax.set_xticklabels(
            [MODEL_LABELS.get(m, m) for m in sub.index],
            rotation=45,
            ha="right",
        )

        ax.set_ylabel(LABELS[parameter])
        ax.set_title(TRUTH_SCENARIOS[scenario_name]["label"])
        ax.grid(True, alpha=0.25)

    fig.suptitle(
        f"Top 5 Models: Recovery of {LABELS[parameter]}",
        y=1.02,
    )

    filename = TOP_PLOT_DIR / f"top5_recovery_{parameter}.png"

    fig.savefig(
        filename,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close(fig)

    print(f"Saved: {filename}")

Top 5 models:
1. Tort2 + kappa + d_soma
2. Full + kappa + d_soma
3. Tort3 + kappa + d_soma
4. Tort1 + kappa + d_soma
Saved: recovery_plots_top5/top5_recovery_r_soma.png
Saved: recovery_plots_top5/top5_recovery_soma_fraction.png
Saved: recovery_plots_top5/top5_recovery_csf_fraction.png


In [107]:
# ============================================================================
# 10. Recovery of d_soma for models where d_soma was free
# ============================================================================

DSOMA_TRUTH = TRUE_BASE["d_soma"]

# Only models/configurations where d_soma was actually free.
DSOMA_MODELS = [
    model_name
    for model_name, spec in MODEL_CONFIGS.items()
    if spec["kwargs"].get("vary_d_soma", False)
]

print("Models with d_soma free:")
for model in DSOMA_MODELS:
    print(f"  {MODEL_LABELS.get(model, model)}")


# ----------------------------------------------------------------------------
# Build d_soma recovery metrics
# ----------------------------------------------------------------------------

d_soma_rows = []

for scenario_name, sub in results_df[
    results_df["model"].isin(DSOMA_MODELS)
].groupby("scenario"):

    for model_name, model_sub in sub.groupby("model"):

        if "d_soma" not in model_sub.columns:
            continue

        values = pd.to_numeric(
            model_sub["d_soma"],
            errors="coerce",
        )

        valid = values.notna()

        if valid.sum() == 0:
            continue

        values_valid = values[valid]
        errors = values_valid - DSOMA_TRUTH

        d_soma_rows.append({
            "scenario": scenario_name,
            "model": model_name,
            "family": MODEL_CONFIGS[model_name]["family"],
            "truth": DSOMA_TRUTH,
            "mean": values_valid.mean(),
            "bias": errors.mean(),
            "mae": errors.abs().mean(),
            "rmse": np.sqrt(np.mean(errors**2)),
            "sd": values_valid.std(ddof=1),
            "relative_rmse": (
                np.sqrt(np.mean(errors**2))
                / abs(DSOMA_TRUTH)
            ),
            "n_valid": int(valid.sum()),
            "n_total": len(values),
        })

d_soma_metrics = pd.DataFrame(d_soma_rows)

display(
    d_soma_metrics.sort_values(
        ["scenario", "rmse"]
    )
)

Models with d_soma free:
  Full + kappa + d_soma
  Tort1 + kappa + d_soma
  Tort2 + kappa + d_soma
  Tort3 + kappa + d_soma


,scenario,model,family,truth,mean,bias,mae,rmse,sd,relative_rmse,n_valid,n_total
0,truth_1,full_soma_diffusivity_free,full_noddi_soma,3.0,3.228306,0.228306,0.228306,0.228306,NaN,0.076102,1,1
2,truth_1,full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,3.0,1.802549,-1.197451,1.197451,1.197451,NaN,0.399150,1,1
1,truth_1,full_tort1_soma_diffusivity_free,full_noddi_soma_tort1,3.0,1.426783,-1.573217,1.573217,1.573217,NaN,0.524406,1,1
3,truth_1,full_tort3_soma_diffusivity_free,full_noddi_soma_tort3,3.0,1.032474,-1.967526,1.967526,1.967526,NaN,0.655842,1,1
5,truth_2,full_tort1_soma_diffusivity_free,full_noddi_soma_tort1,3.0,2.977116,-0.022884,0.022884,0.022884,NaN,0.007628,1,1
4,truth_2,full_soma_diffusivity_free,full_noddi_soma,3.0,1.575722,-1.424278,1.424278,1.424278,NaN,0.474759,1,1
6,truth_2,full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,3.0,1.427946,-1.572054,1.572054,1.572054,NaN,0.524018,1,1
7,truth_2,full_tort3_soma_diffusivity_free,full_noddi_soma_tort3,3.0,1.096421,-1.903579,1.903579,1.903579,NaN,0.634526,1,1
8,truth_3,full_soma_diffusivity_free,full_noddi_soma,3.0,2.855469,-0.144531,0.144531,0.144531,NaN,0.048177,1,1
10,truth_3,full_tort2_soma_diffusivity_free,full_noddi_soma_tort2,3.0,2.219959,-0.780041,0.780041,0.780041,NaN,0.260014,1,1


In [108]:
# ----------------------------------------------------------------------------
# Overall d_soma performance across truth scenarios
# ----------------------------------------------------------------------------

d_soma_summary = (
    d_soma_metrics
    .groupby(
        ["model", "family"]
    )
    .agg(
        mean_d_soma=("mean", "mean"),
        bias=("bias", "mean"),
        mae=("mae", "mean"),
        rmse=("rmse", "mean"),
        sd=("sd", "mean"),
        relative_rmse=("relative_rmse", "mean"),
    )
    .reset_index()
    .sort_values("rmse")
)

d_soma_summary["model"] = d_soma_summary["model"].map(
    lambda x: MODEL_LABELS.get(x, x)
)

display(d_soma_summary)

,model,family,mean_d_soma,bias,mae,rmse,sd,relative_rmse
0,Full + kappa + d_soma,full_noddi_soma,2.553166,-0.446834,0.599038,0.599038,NaN,0.199679
1,Tort1 + kappa + d_soma,full_noddi_soma_tort1,2.119453,-0.880547,0.880547,0.880547,NaN,0.293516
2,Tort2 + kappa + d_soma,full_noddi_soma_tort2,1.816818,-1.183182,1.183182,1.183182,NaN,0.394394
3,Tort3 + kappa + d_soma,full_noddi_soma_tort3,1.088685,-1.911315,1.911315,1.911315,NaN,0.637105


In [92]:
# ============================================================================
# d_soma recovery plot
# ============================================================================

DSOMA_PLOT_DIR = Path("recovery_plots")
DSOMA_PLOT_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5),
    constrained_layout=True,
)

for ax, (scenario_name, scenario) in zip(
    axes,
    TRUTH_SCENARIOS.items(),
):

    sub = (
        d_soma_metrics[
            d_soma_metrics["scenario"] == scenario_name
        ]
        .set_index("model")
        .reindex(DSOMA_MODELS)
        .dropna(subset=["mean"])
    )

    if sub.empty:
        ax.axis("off")
        continue

    x = np.arange(len(sub))

    ax.errorbar(
        x,
        sub["mean"],
        yerr=sub["sd"],
        fmt="o",
        capsize=4,
    )

    ax.axhline(
        DSOMA_TRUTH,
        linestyle="--",
        linewidth=1.5,
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        [
            MODEL_LABELS.get(model, model)
            for model in sub.index
        ],
        rotation=70,
        ha="right",
    )

    ax.set_ylabel(r"$d_{soma}$ ($\mu$m$^2$/ms)")
    ax.set_title(
        TRUTH_SCENARIOS[scenario_name]["label"]
    )
    ax.grid(True, alpha=0.25)

fig.suptitle(
    r"Recovery of $d_{soma}$ for Models with Free Soma Diffusivity",
    y=1.02,
)

filename = DSOMA_PLOT_DIR / "recovery_d_soma.png"

fig.savefig(
    filename,
    dpi=200,
    bbox_inches="tight",
)

plt.close(fig)

print(f"Saved: {filename}")

Saved: recovery_plots/recovery_d_soma.png


## 9. Save results

In [89]:
OUT_DIR=Path("experiment_1b_expanded_no_extra_results")
OUT_DIR.mkdir(exist_ok=True)
results_df.to_csv(OUT_DIR/"raw_fit_results.csv",index=False)
metrics_df.to_csv(OUT_DIR/"primary_recovery_metrics.csv",index=False)
robustness.to_csv(OUT_DIR/"fit_robustness.csv",index=False)
truth_df.to_csv(OUT_DIR/"truth_scenarios.csv",index=False)
family_pivot.to_csv(OUT_DIR/"family_comparison.csv")
print(f"Saved results to: {OUT_DIR.resolve()}")


Saved results to: /Users/Ronan/Documents/PhD/Tools/microstructure-mri-main/examples/notebooks/experiment_1b_expanded_no_extra_results
